<a href="https://colab.research.google.com/github/harshita042/PRML-CSL2050/blob/main/PRML_SVM_lab_script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import required libraries

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from sklearn import svm
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

## Visualize Logistic Regression vs. SVM boundaries and margins on a 2-class dataset


In [ ]:
from sklearn.datasets import make_blobs
from sklearn.linear_model import LogisticRegression

X_sep, y_sep = make_blobs(n_samples=60, centers=2, random_state=6, cluster_std=1.2)

log_reg = LogisticRegression()
log_reg.fit(X_sep, y_sep)

svm_margin = svm.SVC(kernel='linear', C=1000)
svm_margin.fit(X_sep, y_sep)

fig = go.Figure()

fig.add_trace(go.Scatter(x=X_sep[:, 0], y=X_sep[:, 1], mode='markers',
                         marker=dict(color=y_sep, colorscale='RdBu', size=10,
                                     line=dict(width=1, color='black')),
                         name='Data Points'))

x_min, x_max = X_sep[:, 0].min() - 1, X_sep[:, 0].max() + 1
x_vals = np.array([x_min, x_max])

# ---------------------------------------------------------
# Plot Logistic Regression Boundary using exact coefficients
# Equation: w0*x + w1*y + b = 0  =>  y = -(w0*x + b) / w1
# ---------------------------------------------------------
w_log = log_reg.coef_[0]
b_log = log_reg.intercept_[0]
y_vals_log = -(w_log[0] * x_vals + b_log) / w_log[1]

fig.add_trace(go.Scatter(x=x_vals, y=y_vals_log, mode='lines',
                         line=dict(color='green', width=3, dash='dash'),
                         name='Logistic Reg. Boundary'))

w_svm = svm_margin.coef_[0]
b_svm = svm_margin.intercept_[0]
y_vals_svm = -(w_svm[0] * x_vals + b_svm) / w_svm[1]

fig.add_trace(go.Scatter(x=x_vals, y=y_vals_svm, mode='lines',
                         line=dict(color='black', width=4),
                         name='SVM Boundary'))

# SVM Margins (Equations: w0*x + w1*y + b = 1 and -1)
margin_up = -(w_svm[0] * x_vals + b_svm - 1) / w_svm[1]
margin_down = -(w_svm[0] * x_vals + b_svm + 1) / w_svm[1]

fig.add_trace(go.Scatter(x=x_vals, y=margin_up, mode='lines',
                         line=dict(color='gray', width=2, dash='dot'),
                         name='SVM Margin', showlegend=False))
fig.add_trace(go.Scatter(x=x_vals, y=margin_down, mode='lines',
                         line=dict(color='gray', width=2, dash='dot'),
                         name='SVM Margin', showlegend=False))

sv = svm_margin.support_vectors_
fig.add_trace(go.Scatter(x=sv[:, 0], y=sv[:, 1], mode='markers',
                         marker=dict(size=14, color='rgba(0,0,0,0)',
                                     line=dict(width=2, color='gold')),
                         name='Support Vectors'))

fig.update_layout(title="Why SVM? The Maximum Margin (Black) vs. Logistic Regression (Green Dashed)",
                  xaxis_title="Feature 1", yaxis_title="Feature 2",
                  width=850, height=600, template="plotly_white",
                  yaxis=dict(range=[X_sep[:, 1].min() - 1, X_sep[:, 1].max() + 1]), # Locks y-axis bounds
                  legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01))
fig.show()

## Compare LDA vs. SVM boundaries to show how outliers skew LDA but leave SVM robust


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

X_base, y_base = make_blobs(n_samples=50, centers=2, random_state=42, cluster_std=1.2)

outlier_X = np.array([[12, 10], [13, 11], [12.5, 10.5]])
outlier_y = np.array([1, 1, 1])
X_outlier = np.vstack([X_base, outlier_X])
y_outlier = np.concatenate([y_base, outlier_y])

lda = LinearDiscriminantAnalysis()
lda.fit(X_outlier, y_outlier)

svm_outlier = svm.SVC(kernel='linear', C=1.0)
svm_outlier.fit(X_outlier, y_outlier)

fig = go.Figure()

fig.add_trace(go.Scatter(x=X_outlier[:, 0], y=X_outlier[:, 1], mode='markers',
                         marker=dict(color=y_outlier, colorscale='RdBu', size=10,
                                     line=dict(width=1, color='black')),
                         name='Data Points (with Outliers)'))

x_min, x_max = X_outlier[:, 0].min() - 1, X_outlier[:, 0].max() + 1
x_vals = np.array([x_min, x_max])

# ---------------------------------------------------------
# Plot LDA Boundary
# Equation: w0*x + w1*y + b = 0  =>  y = -(w0*x + b) / w1
# ---------------------------------------------------------
w_lda = lda.coef_[0]
b_lda = lda.intercept_[0]
y_vals_lda = -(w_lda[0] * x_vals + b_lda) / w_lda[1]

fig.add_trace(go.Scatter(x=x_vals, y=y_vals_lda, mode='lines',
                         line=dict(color='purple', width=3, dash='dash'),
                         name='LDA Boundary (Skewed by Outliers)'))

w_svm_out = svm_outlier.coef_[0]
b_svm_out = svm_outlier.intercept_[0]
y_vals_svm_out = -(w_svm_out[0] * x_vals + b_svm_out) / w_svm_out[1]

fig.add_trace(go.Scatter(x=x_vals, y=y_vals_svm_out, mode='lines',
                         line=dict(color='black', width=4),
                         name='SVM Boundary (Unaffected)'))

fig.update_layout(title="SVM vs. LDA: How Outliers Destroy LDA",
                  xaxis_title="Feature 1", yaxis_title="Feature 2",
                  width=850, height=600, template="plotly_white",
                  yaxis=dict(range=[X_outlier[:, 1].min() - 1, X_outlier[:, 1].max() + 1]),
                  legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01))
fig.show()

## Generate and visualize a non-linear dataset (circles)

In [ ]:
X, y = make_circles(n_samples=300, factor=0.3, noise=0.1, random_state=42)

fig = px.scatter(x=X[:, 0], y=X[:, 1], color=y.astype(str),
                 color_discrete_sequence=['red', 'blue'],
                 title="Raw Data: Not Linearly Separable",
                 labels={'x': 'Feature 1', 'y': 'Feature 2'})
fig.update_traces(marker=dict(size=8, line=dict(width=1, color='black')))
fig.update_layout(width=700, height=500, template="plotly_white")
fig.show()

## Define a helper function to plot 2D decision boundaries using Plotly


In [ ]:
def plot_decision_boundary_2d(model, X, y, title):
    # Create a meshgrid
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    h = 0.02
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

    # Predict over the grid
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # Create Plotly figure
    fig = go.Figure()

    # Add contour for decision boundary
    fig.add_trace(go.Contour(x=np.arange(x_min, x_max, h), y=np.arange(y_min, y_max, h),
                             z=Z, colorscale='RdBu', opacity=0.4, showscale=False,
                             hoverinfo='skip'))

    # Add scatter plot for data points
    fig.add_trace(go.Scatter(x=X[:, 0], y=X[:, 1], mode='markers',
                             marker=dict(color=y, colorscale='RdBu', size=8,
                                         line=dict(width=1, color='black')),
                             name='Data Points'))

    fig.update_layout(title=title, xaxis_title="Feature 1", yaxis_title="Feature 2",
                      width=700, height=500, template="plotly_white")
    fig.show()

##Train linear SVM on non linearly separable data

In [ ]:
svm_linear = svm.SVC(kernel='linear', C=1.0)
svm_linear.fit(X, y)

plot_decision_boundary_2d(svm_linear, X, y, "SVM with Linear Kernel (Underfitting)")

## Manually map data into 3D with a nonlinear feature and show how a linear SVM separates it with a hyperplane.

In [ ]:
Z_feature = np.exp(-(X[:, 0]**2 + X[:, 1]**2))

svm_3d = svm.SVC(kernel='linear', C=1.0)
X_3d = np.c_[X, Z_feature]
svm_3d.fit(X_3d, y)

xx, yy = np.meshgrid(np.linspace(X[:,0].min(), X[:,0].max(), 50),
                     np.linspace(X[:,1].min(), X[:,1].max(), 50))

# Equation of the plane: w0*x + w1*y + w2*z + b = 0 => z = (-w0*x - w1*y - b) / w2
w = svm_3d.coef_[0]
b = svm_3d.intercept_[0]
zz_plane = (-w[0] * xx - w[1] * yy - b) / w[2]

fig = go.Figure()

fig.add_trace(go.Scatter3d(x=X[:, 0], y=X[:, 1], z=Z_feature, mode='markers',
                           marker=dict(color=y, colorscale='RdBu', size=5,
                                       line=dict(width=1, color='black')),
                           name='Projected Data'))

fig.add_trace(go.Surface(x=xx, y=yy, z=zz_plane, colorscale='Greens', opacity=0.5,
                         name='Separating Hyperplane', showscale=False))

fig.update_layout(title="3D visualization",
                  scene=dict(xaxis_title='Feature 1', yaxis_title='Feature 2', zaxis_title='Transformed Feature Z'),
                  width=800, height=600)
fig.show()

## Train an SVM with RBF kernel

In [ ]:
svm_rbf = svm.SVC(kernel='rbf', C=1.0, gamma='scale')
svm_rbf.fit(X, y)

plot_decision_boundary_2d(svm_rbf, X, y, "SVM with RBF Kernel (Non-linear separation)")

## Train an SVM with polynomial kernel (degree=2)

In [ ]:
svm_poly = svm.SVC(kernel='poly', degree=2, C=1.0, coef0=1)
svm_poly.fit(X, y)

plot_decision_boundary_2d(svm_poly, X, y, "SVM with Polynomial Kernel (Degree=2)")

## Perform hyperparameter tuning for SVM using GridSearchCV and display best parameters


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import make_blobs
import pandas as pd

X_3d, y_3d = make_blobs(n_samples=500, centers=2, n_features=3, cluster_std=2.0, random_state=42)

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [1, 0.1, 0.01, 0.001],
    'kernel': ['rbf']
}

grid = GridSearchCV(svm.SVC(), param_grid, refit=True, verbose=1, cv=5)
grid.fit(X_3d, y_3d)

print(f"Best Parameters: {grid.best_params_}")
print(f"Best Score: {grid.best_score_:.2f}")

results_df = pd.DataFrame(grid.cv_results_)
display(results_df[['param_C', 'param_gamma', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head())

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best Parameters: {'C': 0.1, 'gamma': 0.1, 'kernel': 'rbf'}
Best Score: 1.00


,param_C,param_gamma,mean_test_score
1,0.1,0.100,1.0
2,0.1,0.010,1.0
3,0.1,0.001,1.0
5,1.0,0.100,1.0
7,1.0,0.001,1.0


## Visualize the 3D decision boundary of the best SVM model found via GridSearchCV


In [ ]:
# Create a dense 3D grid to evaluate the model
x_min, x_max = X_3d[:, 0].min() - 1, X_3d[:, 0].max() + 1
y_min, y_max = X_3d[:, 1].min() - 1, X_3d[:, 1].max() + 1
z_min, z_max = X_3d[:, 2].min() - 1, X_3d[:, 2].max() + 1

# Creating a mesh for the 3D space
X_mesh, Y_mesh, Z_mesh = np.mgrid[x_min:x_max:20j, y_min:y_max:20j, z_min:z_max:20j]
grid_3d = np.vstack([X_mesh.ravel(), Y_mesh.ravel(), Z_mesh.ravel()]).T

# Use the best model from GridSearchCV to predict decision values
# decision_function gives the distance to the hyperplane
decision_values = grid.decision_function(grid_3d)
decision_values = decision_values.reshape(X_mesh.shape)

# --- Create 3D Plotly Visualization ---
fig = go.Figure()

# Plot the actual data points
fig.add_trace(go.Scatter3d(
    x=X_3d[:, 0], y=X_3d[:, 1], z=X_3d[:, 2],
    mode='markers',
    marker=dict(color=y_3d, colorscale='RdBu', size=4, opacity=0.8, line=dict(width=1, color='black')),
    name='Data Points'
))

# Plot the Decision Boundary (The Isosurface where decision_function = 0)
fig.add_trace(go.Isosurface(
    x=X_mesh.flatten(),
    y=Y_mesh.flatten(),
    z=Z_mesh.flatten(),
    value=decision_values.flatten(),
    isomin=0,
    isomax=0,
    surface_count=1,
    colorscale='Greens',
    opacity=0.3,
    caps=dict(x_show=False, y_show=False, z_show=False),
    name='Decision Boundary'
))

fig.update_layout(
    title=f"3D Decision Boundary found by GridSearchCV (C={grid.best_params_['C']}, γ={grid.best_params_['gamma']})",
    scene=dict(xaxis_title='Feature 1', yaxis_title='Feature 2', zaxis_title='Feature 3'),
    width=900, height=700
)
fig.show()

# **This is a sample code to start with your task**

```python
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the dataset directly from UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"
data = pd.read_csv(url, sep=';')

# Create a binary classification task: "Good Wine" (7 or higher) vs "Not Good" (below 7)
data['target'] = (data['quality'] >= 7).astype(int)
X = data.drop(['quality', 'target'], axis=1)
y = data['target']

# Split and Scale (Crucial for SVM!)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Dataset Loaded: {X_train.shape[0]} training samples, {X_train.shape[1]} features.")

##1. The "Scaling" Trap:
Train a Linear SVM on the unscaled data (use X_train_raw) and another on the scaled data. Compare the training time and the accuracy. Why did the unscaled version take so much longer to converge?

##2. Kernel Combat:
Find which kernel understands "quality" better. Since we have 11 features use PCA (Principal Component Analysis) to reduce the 11 features to 2 and use the plot_decision_boundary_2d function from our previous cells to visualize each kernel's "shape."

##3. Overfitting vs. Underfitting:
Using the RBF kernel, keep C=1, vary the value of gamma and see at what point does the Training accuracy stay high while the Test accuracy drops.

##4. The Performance Report:
For your best model, generate a Confusion Matrix and a Classification Report

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the dataset directly from UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"
data = pd.read_csv(url, sep=';')

# Create a binary classification task: "Good Wine" (7 or higher) vs "Not Good" (below 7)
data['target'] = (data['quality'] >= 7).astype(int)
X = data.drop(['quality', 'target'], axis=1)
y = data['target']

# Split and Scale (Crucial for SVM!)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Dataset Loaded: {X_train.shape[0]} training samples, {X_train.shape[1]} features.")

Dataset Loaded: 3918 training samples, 11 features.
